# Homework 1
In the first week's homework, the objective is to fit a linear regression model to predict the duration of a New York Yellow Taxi ride. For the homework, we use trip records of yellow taxis from January and February 2023.

We begin by importing the required packages and the data.

In [ ]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [ ]:
df_yellow_jan = pd.read_parquet("../data/yellow_taxi/yellow_tripdata_2023-01.parquet")
df_yellow_feb = pd.read_parquet("../data/yellow_taxi/yellow_tripdata_2023-02.parquet")

## Question 1: Columns in January (D)
The data for January 2023 contains 19 columns.

In [ ]:
len(df_yellow_jan.columns)

## Question 2: Standard deviation of trip duration (B)
For the data belonging to January 2023, the standard deviation of trip durations is 42.59 minutes.

In [ ]:
def process_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the taxi trip dataframe by:
    - Converting pickup and dropoff datetime columns to datetime objects
    - Calculating trip duration in minutes and adding it as a new column
    - Filtering out trips with duration less than 1 minute or greater than 60 minutes

    Args:
        df (pd.DataFrame): Input dataframe with taxi trip records

    Returns:
        pd.DataFrame: Processed dataframe with a 'duration' column and outliers removed
    """
    df.loc[:, "tpep_pickup_datetime"] = pd.to_datetime(
        df["tpep_pickup_datetime"], format="%Y-%m-%d %H:%M:%S"
    )
    df.loc[:, "tpep_dropoff_datetime"] = pd.to_datetime(
        df["tpep_dropoff_datetime"], format="%Y-%m-%d %H:%M:%S"
    )
    df.loc[:, "duration"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    df.loc[:, "duration"] = df["duration"].dt.total_seconds() / 60.0

    return df


In [ ]:
df_yellow_jan_processed = process_data(df_yellow_jan)
df_yellow_jan_processed["duration"].std()

## Question 3: Dropping outliers (D)
After dropping the outliers, 98% of the records are retained.

In [ ]:
orig_jan_count = len(df_yellow_jan)
df_yellow_jan_processed = df_yellow_jan_processed[
    (df_yellow_jan_processed["duration"] >= 1)
    & (df_yellow_jan_processed["duration"] <= 60)
]
filtered_jan_count = len(df_yellow_jan_processed)
filtered_jan_count / orig_jan_count

## Question 4: One-hot encoding (C)
After one-hot encoding the pickup and dropoff locations, the number of columns in the resulting data matrix is 515.

In [ ]:
features = ["PULocationID", "DOLocationID"]
df_yellow_jan_processed.loc[:, features] = df_yellow_jan_processed[features].astype(str)

dv = DictVectorizer()
train_dicts = df_yellow_jan_processed[features].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)
X_train.shape

In [ ]:
dv.feature_names_

## Question 5: Training a model (B)
A vanilla linear regression model with one-hot encoded versions of pickup and dropoff points as features has a RMSE of 7.64. 

In [ ]:
target = "duration"
y_train = df_yellow_jan_processed[target].values

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_train_pred = lr.predict(X_train)

In [ ]:
root_mean_squared_error(y_train, y_train_pred)

In [ ]:
df_yellow_feb_processed = process_data(df_yellow_feb)
df_yellow_feb_processed = df_yellow_feb_processed[
    (df_yellow_feb_processed["duration"] >= 1)
    & (df_yellow_feb_processed["duration"] <= 60)
]

features = ["PULocationID", "DOLocationID"]
df_yellow_feb_processed.loc[:, features] = df_yellow_feb_processed[features].astype(str)
test_dicts = df_yellow_feb_processed[features].to_dict(orient="records")
X_test = dv.transform(test_dicts)
X_test.shape

In [ ]:
y_test_pred = lr.predict(X_test)

target = "duration"
root_mean_squared_error(df_yellow_feb_processed[target].values, y_test_pred)